# 3. Text Preprocessing

In [1]:
from datasets import load_dataset
import pandas as pd
import numpy as np

dataset = load_dataset("dair-ai/emotion")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})


In [2]:
# Converting Dataset Splits to DataFrames

train_df = dataset["train"].to_pandas()
val_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

print("Training shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Training shape: (16000, 2)
Validation shape: (2000, 2)
Test shape: (2000, 2)


In [3]:
#  Checking for Missing Values

print("Training missing values:")
print(train_df.isnull().sum())

print("\nValidation missing values:")
print(val_df.isnull().sum())

print("\nTest missing values:")
print(test_df.isnull().sum())

Training missing values:
text     0
label    0
dtype: int64

Validation missing values:
text     0
label    0
dtype: int64

Test missing values:
text     0
label    0
dtype: int64


In [4]:
#  Checking for Duplicate Records

print("Training duplicate rows:", train_df.duplicated().sum())
print("Validation duplicate rows:", val_df.duplicated().sum())
print("Test duplicate rows:", test_df.duplicated().sum())

Training duplicate rows: 1
Validation duplicate rows: 0
Test duplicate rows: 0


In [5]:
# Inspecting Duplicate Records

duplicate_rows = train_df[
    train_df.duplicated(keep=False)
]

duplicate_rows

,text,label
4975,i feel more adventurous willing to take risks ...,1
13846,i feel more adventurous willing to take risks ...,1


In [6]:
# Removing Duplicate Records

before = len(train_df)

train_df = train_df.drop_duplicates(
    subset=["text", "label"],
    keep="first"
).reset_index(drop=True)

after = len(train_df)

print("Training samples before removing duplicates:", before)
print("Training samples after removing duplicates:", after)
print("Duplicates removed:", before - after)

Training samples before removing duplicates: 16000
Training samples after removing duplicates: 15999
Duplicates removed: 1


In [7]:
#  Verifying Duplicate Removal

print("Remaining duplicate rows:", train_df.duplicated().sum())


Remaining duplicate rows: 0


In [8]:
#  Inspecting Raw Text

pd.set_option("display.max_colwidth", 300)

train_df[["text", "label"]].sample(
    10,
    random_state=42
)

,text,label
8756,ive made it through a week i just feel beaten down,0
4660,i feel this strategy is worthwhile,1
6095,i feel so worthless and weak what does he have to say that s what i want to find out,0
304,i feel clever nov,1
8241,im moved in ive been feeling kind of gloomy,0
15117,i don t recommend carrying around handfuls of gold jewelry in your backpacks or having it on your person i do think that a few key pieces of fun jewelry are necessary to keep you looking and feeling cute on the road,1
1035,i feel confused too,4
9975,when a friend of mine keeps telling me morbid things that happened to his dog,3
7872,i feel like i liked my hair much better before i was using a sulfate free brand and i believe i am using a reputable brand,2
8341,i feel the self pressured expectation to keep up to date with our family events so in order to assuage the guilt here we go,4


In [9]:
#  Checking Text Formatting

print("Empty training texts:", (train_df["text"].str.strip() == "").sum())
print("Leading/trailing whitespace:", 
      (train_df["text"] != train_df["text"].str.strip()).sum())
print("Lowercase texts:", 
      train_df["text"].eq(train_df["text"].str.lower()).sum(),
      "out of", len(train_df))

Empty training texts: 0
Leading/trailing whitespace: 0
Lowercase texts: 15999 out of 15999


In [10]:
#  Defining Safe Text Normalization

import re

def normalize_text(text):
    text = str(text)
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

In [11]:
train_df["clean_text"] = train_df["text"].apply(normalize_text)
val_df["clean_text"] = val_df["text"].apply(normalize_text)
test_df["clean_text"] = test_df["text"].apply(normalize_text)

print("Training columns:", train_df.columns.tolist())
print("Validation columns:", val_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())

Training columns: ['text', 'label', 'clean_text']
Validation columns: ['text', 'label', 'clean_text']
Test columns: ['text', 'label', 'clean_text']


In [12]:
#  Comparing Original and Cleaned Text

comparison_df = train_df[
    ["text", "clean_text"]
].sample(
    10,
    random_state=42
)

comparison_df

,text,clean_text
8756,ive made it through a week i just feel beaten down,ive made it through a week i just feel beaten down
4660,i feel this strategy is worthwhile,i feel this strategy is worthwhile
6095,i feel so worthless and weak what does he have to say that s what i want to find out,i feel so worthless and weak what does he have to say that s what i want to find out
304,i feel clever nov,i feel clever nov
8241,im moved in ive been feeling kind of gloomy,im moved in ive been feeling kind of gloomy
15117,i don t recommend carrying around handfuls of gold jewelry in your backpacks or having it on your person i do think that a few key pieces of fun jewelry are necessary to keep you looking and feeling cute on the road,i don t recommend carrying around handfuls of gold jewelry in your backpacks or having it on your person i do think that a few key pieces of fun jewelry are necessary to keep you looking and feeling cute on the road
1035,i feel confused too,i feel confused too
9975,when a friend of mine keeps telling me morbid things that happened to his dog,when a friend of mine keeps telling me morbid things that happened to his dog
7872,i feel like i liked my hair much better before i was using a sulfate free brand and i believe i am using a reputable brand,i feel like i liked my hair much better before i was using a sulfate free brand and i believe i am using a reputable brand
8341,i feel the self pressured expectation to keep up to date with our family events so in order to assuage the guilt here we go,i feel the self pressured expectation to keep up to date with our family events so in order to assuage the guilt here we go


In [13]:
#  Validating the Cleaned Text

print("Training cleaned text missing:", train_df["clean_text"].isnull().sum())
print("Validation cleaned text missing:", val_df["clean_text"].isnull().sum())
print("Test cleaned text missing:", test_df["clean_text"].isnull().sum())

print("\nTraining empty cleaned texts:",
      (train_df["clean_text"].str.strip() == "").sum())

print("Validation empty cleaned texts:",
      (val_df["clean_text"].str.strip() == "").sum())

print("Test empty cleaned texts:",
      (test_df["clean_text"].str.strip() == "").sum())

print("\nTraining duplicate text-label pairs:",
      train_df.duplicated(subset=["clean_text", "label"]).sum())

print("Validation duplicate text-label pairs:",
      val_df.duplicated(subset=["clean_text", "label"]).sum())

print("Test duplicate text-label pairs:",
      test_df.duplicated(subset=["clean_text", "label"]).sum())

Training cleaned text missing: 0
Validation cleaned text missing: 0
Test cleaned text missing: 0

Training empty cleaned texts: 0
Validation empty cleaned texts: 0
Test empty cleaned texts: 0

Training duplicate text-label pairs: 0
Validation duplicate text-label pairs: 0
Test duplicate text-label pairs: 0


In [14]:
#  Final Preprocessing Check

print("Training dataset:")
print(train_df.shape)
print(train_df.columns.tolist())

print("\nValidation dataset:")
print(val_df.shape)
print(val_df.columns.tolist())

print("\nTest dataset:")
print(test_df.shape)
print(test_df.columns.tolist())

print("\nSample processed records:")
train_df[["text", "clean_text", "label"]].head()

Training dataset:
(15999, 3)
['text', 'label', 'clean_text']

Validation dataset:
(2000, 3)
['text', 'label', 'clean_text']

Test dataset:
(2000, 3)
['text', 'label', 'clean_text']

Sample processed records:


,text,clean_text,label
0,i didnt feel humiliated,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake,i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake,0
2,im grabbing a minute to post i feel greedy wrong,im grabbing a minute to post i feel greedy wrong,3
3,i am ever feeling nostalgic about the fireplace i will know that it is still on the property,i am ever feeling nostalgic about the fireplace i will know that it is still on the property,2
4,i am feeling grouchy,i am feeling grouchy,3


In [15]:
#  Saving the Processed Datasets

import os

os.makedirs("../data/processed", exist_ok=True)

train_df.to_csv(
    "../data/processed/train_processed.csv",
    index=False
)

val_df.to_csv(
    "../data/processed/validation_processed.csv",
    index=False
)

test_df.to_csv(
    "../data/processed/test_processed.csv",
    index=False
)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.


In [16]:
print("Training file:", "../data/processed/train_processed.csv")
print("Validation file:", "../data/processed/validation_processed.csv")
print("Test file:", "../data/processed/test_processed.csv")

Training file: ../data/processed/train_processed.csv
Validation file: ../data/processed/validation_processed.csv
Test file: ../data/processed/test_processed.csv


In [17]:
import os

processed_files = [
    "../data/processed/train_processed.csv",
    "../data/processed/validation_processed.csv",
    "../data/processed/test_processed.csv"
]

for file_path in processed_files:
    print(
        os.path.basename(file_path),
        "->",
        "Exists" if os.path.exists(file_path) else "NOT FOUND"
    )

train_processed.csv -> Exists
validation_processed.csv -> Exists
test_processed.csv -> Exists


## 3.17 Text Preprocessing Summary

The text preprocessing stage has been completed successfully.

### Key preprocessing steps

1. Loaded the Emotion dataset and converted the train, validation, and test splits into Pandas DataFrames.
2. Checked for missing values and confirmed that no missing records were present.
3. Identified one duplicate record in the training dataset and removed it.
4. Verified that no duplicate records remained.
5. Inspected the raw text and identified contractions and negation-related expressions that may contain useful emotional information.
6. Confirmed that the dataset was already lowercase and contained no empty or leading/trailing whitespace records.
7. Applied conservative text normalization by stripping surrounding whitespace and normalizing repeated whitespace.
8. Preserved the original `text` column and created a separate `clean_text` column.
9. Validated the processed datasets for missing, empty, and duplicate cleaned records.
10. Saved the processed datasets for the feature engineering stage.

### Final Dataset Sizes

- Training: 15,999 samples
- Validation: 2,000 samples
- Test: 2,000 samples

### Output Files

- `data/processed/train_processed.csv`
- `data/processed/validation_processed.csv`
- `data/processed/test_processed.csv`

The processed datasets are now ready for feature engineering.